# Module 06 — UDFs & Planner Visibility

In [ ]:
import sys
sys.path.insert(0, '/workspace')
from shared.spark_session import begin_module, ui_links, explain_plan
from shared import config
from shared.utils import timed_action, timed_read, drop_iceberg_table, reset_spark_configs
from shared.demo_data import user_post_activity

spark = begin_module('module-06-udfs')
ui_links()

In [ ]:
from pyspark.sql.functions import udf, col, when, length
from pyspark.sql.types import DoubleType
trips = spark.read.parquet(*config.yellow_taxi_paths())
df = trips.select('PULocationID', 'total_amount').filter('PULocationID IS NOT NULL').limit(500_000)

## Python UDF — opaque to optimizer

In [ ]:
@udf(DoubleType())
def fare_tax_py(amount):
    return float(amount) * 1.08 if amount else 0.0

ud = df.withColumn('fare_taxed', fare_tax_py(col('total_amount')))
ud.explain('formatted')  # BatchEvalPython / no predicate pushdown through UDF

## Native expression — same logic, optimizable

In [ ]:
native = df.withColumn('fare_taxed', col('total_amount') * 1.08)
native.filter('fare_taxed > 50').explain('formatted')

## Benchmark

In [ ]:
_, t_udf = timed_action(ud.select('fare_taxed'))
_, t_native = timed_action(native.select('fare_taxed'))
print(f'Python UDF: {t_udf:.2f}s')
print(f'native:     {t_native:.2f}s')

## Pandas UDF (Arrow vectorized)

In [ ]:
import pandas as pd
from pyspark.sql.functions import pandas_udf

@pandas_udf(DoubleType())
def fare_tax_pandas(s: pd.Series) -> pd.Series:
    return s * 1.08

pd_df = df.withColumn('fare_taxed', fare_tax_pandas(col('total_amount')))
_, t_pd = timed_action(pd_df.select('fare_taxed'))
print(f'pandas UDF: {t_pd:.2f}s')

## Takeaways

1. UDFs block many optimizer rules — compare explain trees.
2. Cost = Python round-trip per batch (or per row) + serialization.
3. Prefer native functions; Pandas UDF when vectorized Python is required.